# 📖 orderbook — free historical order books: one day from OKX, Bybit and IEX, side by side

Downloads ONE day of free level-2 order-book history from three sources, decodes each into the same stream of book messages, replays it into a book, and puts the books next to each other: the real file formats, message rates, depths and defects, before any strategy is chosen.

| source | what the file holds | the day | used here |
|---|---|---|---|
| `okx` | BTC-USDT spot, 400 levels a side: a snapshot every 15 minutes, then an update whenever the book changes (about every 10 ms); each level is `[price, size, number of orders]` | ~130 MB zipped | all of it |
| `bybit` | BTCUSDT spot, the exchange's websocket stream: 200 levels a side, a snapshot, then deltas with an update id | ~60 MB zipped | all of it |
| `iex` | the IEX exchange alone (2-3% of US volume): price-level updates of EVERY US stock, DEEP 1.0 packets in a pcap(ng) capture | 3-15 GB gzipped | the first `IEX_MB` MB: the start of the day, 08:00 New York |

Run it **top to bottom**: 1-2 minutes the first time (about 290 MB to download), seconds once the files are stored. `MARKET = "perp"` switches OKX to the BTC-USDT swap and Bybit to the USDT perpetual (about 0.8 GB more). Code: `orderbook_free.py` (tested in `tests/test_orderbook_free.py`). Data: `var/orderbook_free/` (git-ignored; personal research use only, and IEX's historical data is under IEX's terms of use).

Nothing here is a trading result. The question is only: *what do these free files really contain?*

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import plots, strategies

_spec = importlib.util.spec_from_file_location(
    "orderbook_free", strategies.STRATEGIES_DIR / "orderbook" / "orderbook_free.py"
)
ob = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ob)

NY = "America/New_York"
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 40)


def below(fig, y=-0.15):
    """Legend under the plot."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


def wall(index):
    """Time axis as text: plotly shows text as written, but a number in the viewer's zone."""
    return index.strftime("%Y-%m-%d %H:%M:%S")


# --- 0 · the knobs --------------------------------------------------------------------------------
DAY = None  # 'YYYY-MM-DD'; None = the newest trading day IEX has published
MARKET = "spot"  # "spot", or "perp": OKX swap + Bybit USDT perpetual (~0.8 GB more)
OKX_INST = "BTC-USDT"  # the pair; for "perp" OKX wants the same text as the instrument family
BYBIT_SYMBOL = "BTCUSDT"
IEX_SYMBOLS = ["SPY", "AAPL", "NVDA"]  # the IEX stocks whose books are rebuilt
IEX_MB = 100  # MB of the day's gzip; 100 MB of a recent day reach about 1 minute after the open
GRID_S = 10  # seconds between samples of the crypto books (IEX: every second)
PROFILE_AT = "09:30:30"  # New York time of the depth picture (Fig 2): just after the US open
PROFILE_BPS = 50  # how far from the mid Fig 2 looks

## 1 · One day from each source

* **OKX**: the public history API (`/api/v5/public/market-data-history`, no key) names the day's file: one `.tar.gz` per instrument and UTC day.
* **Bybit**: a plain URL per symbol and UTC day (`quote-saver.bycsi.com/orderbook/spot/BTCUSDT/<day>_BTCUSDT_ob200.data.zip`).
* **IEX**: IEX's file index (`iextrading.com/api/1.0/hist`) lists one capture per trading day for the WHOLE market, 3-15 GB. Only the first `IEX_MB` MB are fetched (an HTTP Range request): the start of the day, so every book is built from empty.

Stored files are never fetched again. `DAY = None` takes the newest day IEX has published; that is a trading day, so all three sources have it.

In [ ]:
DAY = DAY or ob.latest_iex_day()
okx_market, bybit_market = {"spot": ("spot", "spot"), "perp": ("swap", "linear")}[MARKET]
files = {
    "okx": ob.fetch_okx(DAY, OKX_INST, okx_market),
    "bybit": ob.fetch_bybit(DAY, BYBIT_SYMBOL, bybit_market),
    "iex": ob.fetch_iex(DAY, "DEEP", IEX_MB),
}
display(
    pd.DataFrame(
        [
            {
                "source": k,
                "file": v["path"].name,
                "MB": round(v["bytes"] / 1e6, 1),
                "seconds": round(v["seconds"], 1),
                "cached": v["cached"],
            }
            for k, v in files.items()
        ]
    ).set_index("source")
)
print(
    f"day {DAY}: IEX's file for the day is {files['iex']['full_bytes'] / 1e9:.1f} GB, "
    f"we keep the first {IEX_MB} MB"
)

## 2 · What is in the files

OKX and Bybit store JSON lines: the first few are printed raw, cut at 220 characters (the first line is a snapshot). IEX's capture is binary, so it is decoded (about 5 seconds): the table counts its messages by type and shows the first message of each type.

In [ ]:
for k in ("okx", "bybit"):
    print(f"--- {k}: {files[k]['path'].name}")
    for line in ob.raw_head(files[k]["path"], 3):
        print("  ", line)
iex = ob.decode_iex(files["iex"]["path"], IEX_SYMBOLS)
first = pd.Timestamp(iex["first_ns"], tz="UTC").tz_convert(NY).strftime("%H:%M:%S")
last = pd.Timestamp(iex["last_ns"], tz="UTC").tz_convert(NY).strftime("%H:%M:%S")
print(f"--- iex: {files['iex']['path'].name} ({iex['format']} capture, decoded)")
print(
    f"price-level updates from {first} to {last} New York; "
    f"{iex['symbols']:,} stocks had at least one"
)
display(
    pd.DataFrame(
        [
            {"type": t, "name": ob.TYPE_NAMES.get(ord(t), "?"), "messages": n}
            for t, n in iex["types"].most_common()
        ]
    )
)
examples = pd.DataFrame([ob.describe_iex_message(m) for _, m in sorted(iex["examples"].items())])
display(examples.fillna(""))

**The three formats side by side** (measured on the 2026-10-16 files):

| | OKX | Bybit | IEX |
|---|---|---|---|
| container | `.tar.gz` → one `.data` file | `.zip` → one `.data` file | `.pcap.gz` → Ethernet / IPv4 / UDP / IEX-TP packets (classic pcap in 2017, pcapng in every later year checked) |
| a record | one JSON line: `action` (`snapshot` / `update`), `ts`, `asks`, `bids` | one JSON line: `topic`, `ts`, `cts`, `type` (`snapshot` / `delta`), `data` with `b`, `a`, `u`, `seq` | one binary message: Price Level Update, buy `8` or sell `5` (30 bytes) |
| a level | `[price, size, orders]`, as text | `[price, size]`, as text | size `uint32`, price `int64` ÷ 10 000 |
| size 0 | the level is gone | the level is gone | the level is gone |
| time | `ts`, milliseconds | `ts` and `cts` (the matching engine's), milliseconds | nanoseconds |
| snapshots | the first line, then one every 15 minutes (96 a day) | the first line, then rarely (2 a day) | none: the books start empty at 08:00 |
| sequence check | none in the file | `u` goes up by one | IEX-TP packet sequence numbers (no gap in the 100 MB prefix) |
| trades | not in the file | not in the file | in the same capture (Trade Report, type `T`) |
| consistent after | every line | every line | the update flagged as the last of its event |

## 3 · Replay: the same measures for every source

Each stream is applied message by message to an empty book (size 0 deletes a level, a snapshot replaces the book), the best bid and ask are tracked after every message, and the book is sampled on a clock grid (`GRID_S` seconds for crypto, one second for IEX). The OKX day is several million messages and takes 10-30 seconds; nothing is kept but the samples and a few counters.

In [ ]:
open_utc = pd.Timestamp(f"{DAY} 09:30:00", tz=NY).tz_convert("UTC")
at = pd.Timestamp(f"{DAY} {PROFILE_AT}", tz=NY).tz_convert("UTC").value
okx_key, bybit_key = f"okx {OKX_INST}", f"bybit {BYBIT_SYMBOL}"
runs = {
    okx_key: ob.replay(ob.iter_okx(files["okx"]["path"]), grid_s=GRID_S, at_ns=[at]),
    bybit_key: ob.replay(ob.iter_bybit(files["bybit"]["path"]), grid_s=GRID_S, at_ns=[at]),
}
for s, events in iex["events"].items():
    if events:
        runs[f"iex {s}"] = ob.replay(events, grid_s=1, at_ns=[at])
colors = dict(zip(runs, [plots.SERIES[i] for i in (0, 1, 2, 4, 5, 6, 7)], strict=False))
summary = ob.summary(runs)
display(summary)

**Reading the table.** One column per book. For IEX every stock has its own book and a "message" is an *event* (the price-level updates IEX flags as one change).

* `msgs_per_s_*` and `levels_per_msg_*`: how often the book changes and how much one message carries. OKX publishes every change as it happens; Bybit batches them (about 10 messages a second), so each of its messages sets more levels.
* `pct_changes_that_delete`: the share of level changes that remove a level (size 0). A third or more: the edges of the book churn.
* `first_snapshot_levels`: levels in the first snapshot, both sides. IEX has none.
* `spread_bps_*`, `depth10bps_usd_p50`: from the sampled books. A major crypto pair trades at a ONE-TICK spread (0.1 USD on ~80,000 is 0.012 bps), so its spread is a constant and carries no information; depth is where the books differ. The IEX figures mix the thin pre-market with the open (next table).
* The last three rows are defects: a bid at or above the ask, an update-id jump (Bybit only), a message earlier than the one before it. All should be 0.

In [ ]:
rows = []
phases = (("08:00-09:30", slice(None, open_utc)), ("from 09:30", slice(open_utc, None)))
for k, r in runs.items():
    if k.startswith("iex"):
        for label, cut in phases:
            part = r["samples"].loc[cut]
            rows.append(
                {
                    "book": k,
                    "period": label,
                    "seconds sampled": len(part),
                    "spread_bps_p50": part["spread_bps"].median(),
                    "depth10bps_usd_p50": (part["bid_usd_10"] + part["ask_usd_10"]).median(),
                }
            )
display(pd.DataFrame(rows).set_index(["book", "period"]))

## 4 · Pictures

Each book keeps one color in every figure: OKX blue, Bybit green, then the IEX stocks (amber, purple, red). **Fig 1** how often each crypto book changes. **Fig 2** the money resting at each distance from the mid at `PROFILE_AT`, a few seconds after the US open (solid line = bids, dotted = asks). **Fig 3** the depth through the UTC day and how far apart the two crypto mids are. **Fig 4** IEX's spread from 08:00 to just after the open.

In [ ]:
fig = go.Figure()
for k in (okx_key, bybit_key):
    t = runs[k]["times"]
    per_min = np.bincount(((t - t[0]) // (60 * 10**9)).astype(int))
    x = pd.to_datetime(t[0] + np.arange(len(per_min)) * 60 * 10**9, unit="ns", utc=True)
    fig.add_trace(
        go.Scatter(
            x=wall(x),
            y=per_min,
            mode="lines",
            name=f"{k}: messages per minute",
            line={"color": colors[k]},
        )
    )
fig.update_yaxes(title="messages per minute")
plots._base(fig, title=f"Fig 1 · how often each book changes ({DAY}, UTC)", height=380)
below(fig).show()

In [ ]:
want = [okx_key, bybit_key, f"iex {IEX_SYMBOLS[0]}"]
profile = [k for k in want if k in runs and at in runs[k]["books"] and all(runs[k]["books"][at])]
if profile != want:
    print(f"no two-sided book at {PROFILE_AT} New York for {[k for k in want if k not in profile]}")
if profile:
    fig = make_subplots(
        rows=1, cols=len(profile), subplot_titles=profile, horizontal_spacing=0.07
    )
    for c, k in enumerate(profile, start=1):
        curves = ob.depth_profile(*runs[k]["books"][at])
        for curve, side, dash in zip(curves, ("bids", "asks"), ("solid", "dot"), strict=True):
            curve = curve[curve["bps"] <= PROFILE_BPS]
            fig.add_trace(
                go.Scatter(
                    x=curve["bps"],
                    y=curve["usd"],
                    mode="lines",
                    name=f"{k}: {side}",
                    line={"color": colors[k], "shape": "hv", "dash": dash},
                ),
                1,
                c,
            )
        fig.update_xaxes(title="bps from the mid", row=1, col=c)
        fig.update_yaxes(title="cumulative USD", row=1, col=c)
    title = f"Fig 2 · money resting by distance from the mid, {PROFILE_AT} New York"
    plots._base(fig, title=title, height=420)
    below(fig).show()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "USD resting within 10 bps of the mid, both sides (median per 15 minutes)",
        "OKX mid minus Bybit mid, bps of the mid (median per 15 minutes)",
    ),
)
for k in (okx_key, bybit_key):
    smp = runs[k]["samples"].resample("15min").median()
    fig.add_trace(
        go.Scatter(
            x=wall(smp.index),
            y=smp["bid_usd_10"] + smp["ask_usd_10"],
            mode="lines",
            name=f"{k}: USD within 10 bps",
            line={"color": colors[k]},
        ),
        1,
        1,
    )
a, b = runs[okx_key]["samples"]["mid"], runs[bybit_key]["samples"]["mid"]
gap = ((a - b) / b * 1e4).dropna().resample("15min").median()
fig.add_trace(
    go.Scatter(
        x=wall(gap.index),
        y=gap,
        mode="lines",
        name="OKX mid minus Bybit mid (bps)",
        line={"color": plots._INK},
    ),
    2,
    1,
)
plots._base(fig, title=f"Fig 3 · depth and the gap between the two venues ({DAY}, UTC)", height=560)
below(fig, -0.1).show()

In [ ]:
fig = go.Figure()
for k in (k for k in runs if k.startswith("iex")):
    y = plots._thin(runs[k]["samples"]["spread_bps"])
    fig.add_trace(
        go.Scatter(
            x=y.index.tz_convert(NY).strftime("%Y-%m-%d %H:%M:%S"),
            y=y,
            mode="lines",
            name=f"{k}: spread (bps of the mid)",
            line={"color": colors[k]},
        )
    )
o = open_utc.tz_convert(NY).strftime("%Y-%m-%d %H:%M:%S")
fig.add_trace(
    go.Scatter(
        x=[o, o],
        y=[0.1, 3000],
        mode="lines",
        name="09:30 US open",
        line={"color": "#898781", "dash": "dot"},
    )
)
fig.update_yaxes(type="log", title="spread, bps of the mid (log scale)")
title = f"Fig 4 · IEX spreads, 08:00 New York to just after the open ({DAY})"
plots._base(fig, title=title, height=420)
below(fig).show()

## 5 · Do the two crypto books agree?

The same instrument on two venues should show the same price. The mids of the two sampled books on their common grid: the median difference and the median and 95th-percentile absolute difference in bps of the mid, and the correlation of their 1-minute returns. A clock offset between the two files, or a stale feed, shows here first.

In [ ]:
cmp_mid = ob.compare_mid(runs[okx_key]["samples"], runs[bybit_key]["samples"])
display(pd.Series(cmp_mid, name="OKX vs Bybit").to_frame())

## 6 · Notes from building this (the 2026-10-16 files)

* **OKX** BTC-USDT spot: 3.9 million lines in the day (45 a second on average, 82 in the busiest 5% of seconds), 96 snapshots, 400 levels a side all day. **Bybit** BTCUSDT spot: 0.8 million lines (9.4 a second), 2 snapshots, no update-id gap, never silent for more than 2 seconds.
* Both are **one-tick** markets (the spread was one tick, 0.012 bps, in at least 95% of the samples). OKX shows about twice Bybit's depth within 10 bps (median 7.4 vs 3.4 million USD). Their mids differ by a median 0.45 bps and their 1-minute returns correlate 0.994: the two files agree on the price.
* **IEX** price-level updates start at 08:00:00 New York. About 70% of the price-level updates and trades in the first 100 MB fall in the single minute 09:30: the opening burst, when every stock's book is rebuilt (roughly 1 MB of gzip per second). At 09:30:00 SPY's book holds just two placeholder quotes (a bid near 750, an ask near 796); within a second it is a real book again. IEX's books are sparse before the open: over the 100 MB prefix, almost all pre-market, the median spread was 1.3 bps for SPY, 5.1 for NVDA and 17.6 for AAPL, and SPY went as long as 35 minutes without an update.
* **No trades** in the OKX and Bybit order-book files: they are separate downloads (OKX history API module 1, Bybit `public.bybit.com`). IEX's capture carries its trade reports.
* Sizes: one OKX day is 45-525 MB zipped (spot to swap), Bybit 30-380 MB, IEX 3-15 GB for the whole market. A symbol-year of OKX or Bybit is 10-200 GB zipped: choose symbols and windows before downloading in bulk.

## 7 · Checks

In [ ]:
S = summary
checks = [
    ("all three files downloaded", all(f["bytes"] > 0 for f in files.values())),
    (
        "OKX and Bybit cover the whole UTC day",
        bool((S.loc["span_hours", [okx_key, bybit_key]] > 23.9).all()),
    ),
    ("no crossed book in any source", bool((S.loc["crossed_books"] == 0).all())),
    ("Bybit update ids have no gap", bool(S.loc["update_id_gaps", bybit_key] == 0)),
    ("no message earlier than the one before it", bool((S.loc["out_of_order"] == 0).all())),
    (
        "OKX and Bybit mids agree (median |difference| < 2 bps)",
        bool(cmp_mid["median_abs_diff_bps"] < 2),
    ),
    ("every IEX stock has events", all(len(iex["events"][s]) > 0 for s in IEX_SYMBOLS)),
]
display(pd.DataFrame(checks, columns=["check", "ok"]))